In [ ]:
from retrieval.index import DistributedIndex, load_or_initialize_index, build_index
import yaml
import os
from pathlib import Path
import torch
import json
import ir_datasets
import numpy as np
import mteb

In [ ]:
def load_model_meta_yaml(file_path : str | Path) -> dict:
    with open(file_path, "r") as f:
        return yaml.safe_load(f)

In [ ]:
path = Path("/home/rjha5/603-nvme2/arena/model_meta.yml")

In [ ]:
model_meta = load_model_meta_yaml(path)


In [ ]:
[model for model in model_meta["model_meta"].keys() if model_meta["model_meta"][model].get("size", 7000) < 2000]

In [ ]:
from models import ModelManager


model_manager = ModelManager(model_meta=model_meta)

In [ ]:
model = model_manager.load_model("intfloat/multilingual-e5-small")

In [ ]:
model_manager.load_local_index(model_name="intfloat/multilingual-e5-small", corpus="wikipedia", embedbs=1024)

# FOOBAR

In [ ]:
# write dummy passages to a jsonl file
with open("dummy_passages.jsonl", "w") as f:
    for i in range(100):
        f.write(json.dumps({"_id": i, "title": f"Dummy Passage {i}", "text": f"This is a dummy passage {i}"}) + "\n")

In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
# model_name = 'intfloat/multilingual-e5-small'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = mteb.get_model(model_name, revision=model_meta["model_meta"][model_name].get('revision', None), device=device)


In [ ]:
index, passages = load_or_initialize_index(dim=384, passages=["dummy_passages.jsonl"])

In [ ]:
build_index(model.bfloat16(), index, [p["text"] for p in passages], gpu_embedder_batch_size=256)

In [ ]:
index.embeddings

In [ ]:
emb_normed = torch.nn.functional.normalize(index.embeddings, p=2, dim=1)
emb_normed.norm(dim=1)

# Model Manager

In [ ]:
model_meta

In [ ]:
model_name = "nomic-ai/nomic-embed-text-v1.5"
model_name = "BAAI/bge-large-en-v1.5"
model = mteb.get_model(model_name, revision=model_meta["model_meta"][model_name].get('revision', None), device=device)

In [ ]:
nomic = mteb.get_model("nomic-ai/nomic-embed-text-v1.5")
nomic

In [ ]:
hasattr(nomic, "encode_corpus")

In [ ]:
x = nomic.encode_corpus(["foobar", "baz"], convert_to_tensor=False)
type(x), x

In [ ]:
y = nomic.encode_corpus(["foobar", "baz"], convert_to_tensor=True)
type(y), y

In [ ]:
z = nomic.encode_corpus(["foobar", "baz"])
type(z), z

In [ ]:
model_meta["model_meta"][model_name]


In [ ]:
from mteb import Encoder
from typing import Any

from retrieval.index import DTYPE_TO_TORCH_DTYPE

def index_collection(model : Encoder, collection : list[str], model_meta : dict[str, Any] = {}, batch_size=32) -> DistributedIndex:
    
    index = DistributedIndex(dtype=DTYPE_TO_TORCH_DTYPE[model_meta.get("index_dtype", "float32")])
    index.init_embeddings(collection, dim=model_meta["dim"])

    print(index.embeddings.dtype)

    build_index(model, index, collection, gpu_embedder_batch_size=batch_size)

    return index

In [ ]:
index = index_collection(model, collection=[f"foobar the {i}th was a mighty king" for i in range(1000)], model_meta=model_meta["model_meta"][model_name], batch_size=64)

In [ ]:
index.search_knn(model.encode(["foobar doc 1", "foobar doc 42"], convert_to_tensor=True), topk=5)

In [ ]:
import datasets
from tqdm import tqdm

In [ ]:
wikipedia = datasets.load_dataset("orionweller/wikipedia-2024-06-24-docs", split="train")

In [ ]:
wikipedia

In [ ]:
title_text = [f"{title}\n{text}" for title, text in tqdm(zip(wikipedia["title"], wikipedia["text"]), total=len(wikipedia))]

In [ ]:
index = index_collection()

In [ ]:
from retrieval.common import load_passages_from_hf

In [ ]:
wiki = load_passages_from_hf("wikipedia", limit=None)

In [ ]:
wiki

# ST TEST

In [1]:
# Import the main class from the library
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

In [3]:
model_1 = SentenceTransformer("intfloat/multilingual-e5-small", trust_remote_code=True, device="cuda:0")
model_2 = SentenceTransformer("intfloat/multilingual-e5-small", trust_remote_code=True, device="cuda:1")
model_3 = SentenceTransformer("intfloat/multilingual-e5-small", trust_remote_code=True, device="cuda:2")
model_4 = SentenceTransformer("intfloat/multilingual-e5-small", trust_remote_code=True, device="cuda:3")


In [4]:
wikipedia = load_dataset("orionweller/wikipedia-2024-06-24-docs", split="train")

In [5]:
shards = [wikipedia["text"][i:i+100_000] for i in range(0, len(wikipedia["text"]), 100_000)]

In [ ]:
from concurrent.futures import ThreadPoolExecutor
# encode the first 4 shards in parallel

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(model.encode, shard, convert_to_tensor=True, batch_size=1024, show_progress_bar=True) for model, shard in zip([model_1, model_2, model_3, model_4], shards[:4])]
    results = [future.result() for future in futures]



Batches:   0%|          | 0/98 [00:00<?, ?it/s]

Batches:   0%|          | 0/98 [00:00<?, ?it/s]

Batches:   0%|          | 0/98 [00:00<?, ?it/s]

Batches:   0%|          | 0/98 [00:00<?, ?it/s]

In [ ]:

# 1. Define the list of models to test
model_ids = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'intfloat/multilingual-e5-small',
    'intfloat/multilingual-e5-large-instruct',
    'BAAI/bge-large-en-v1.5',
    'jinaai/jina-embeddings-v2-base-en',
    'mixedbread-ai/mxbai-embed-large-v1',
    'nomic-ai/nomic-embed-text-v1.5',
    'nomic-ai/nomic-embed-text-v1'
]

# 2. Create sample queries and documents
queries = [
    "What is the latest news on AI?",
    "How to bake a chocolate cake?",
    "Compare python vs. javascript for web development.",
    "What are the best hiking trails in Maryland?",
    "A summary of the novel 'Dune'."
]

# For instruction-following models, it's good practice to add a prefix.
# We will encode documents as-is, but you could prefix them if needed.
documents = [
    "Artificial intelligence continues to evolve, with new breakthroughs in large language models and generative art happening daily.",
    "To bake a chocolate cake, you first need to preheat your oven to 350°F (175°C). Then, mix flour, sugar, cocoa powder, baking soda, and salt. In a separate bowl, combine eggs, milk, oil, and vanilla extract. Gradually mix the wet ingredients into the dry ones until the batter is smooth.",
    "Python, with frameworks like Django and Flask, is known for its robust backend capabilities and readability. JavaScript, with Node.js, allows for full-stack development using a single language and excels in building real-time, event-driven applications.",
    "Maryland offers diverse hiking experiences, from the Appalachian Trail sections in the west to the scenic coastal paths at Assateague Island. Cunningham Falls State Park and Patapsco Valley State Park are also popular choices for their waterfalls and river views.",
    "'Dune' by Frank Herbert is a science fiction epic set in the distant future on the desert planet Arrakis. It follows the story of Paul Atreides as his noble family accepts stewardship of the planet, the only source of the valuable spice 'melange', leading to political intrigue, betrayal, and rebellion."
]

# 3. Loop through each model and test encoding
for model_id in model_ids:
    print(f"\n{'='*50}")
    print(f"🚀 Testing model: {model_id}")
    print(f"{'='*50}")

    try:
        # Load the model
        model = SentenceTransformer(model_id, trust_remote_code=True, device="cpu")
        print(f"Model: {model}")
        print(f"Model device: {model.device}")

        # 4. Encode the queries and documents
        print("Encoding queries...")
        query_embeddings = model.encode(queries, show_progress_bar=True, convert_to_tensor=True)
        print(f"Shape of query embeddings: {query_embeddings.shape}, dtype: {query_embeddings.dtype}, device: {query_embeddings.device}")
        print(f"Query embeddings norm: {query_embeddings.norm(dim=1)}")

        print("\nEncoding documents...")
        doc_embeddings = model.encode(documents, show_progress_bar=True, convert_to_tensor=True)
        print(f"Shape of document embeddings: {doc_embeddings.shape}, dtype: {doc_embeddings.dtype}, device: {doc_embeddings.device}")
        print(f"Document embeddings norm: {doc_embeddings.norm(dim=1)}")

        # 5. Print the results to verify
        print("\n✅ Encoding successful!")
        
    except Exception as e:
        print(f"\n❌ Failed to load or encode with model {model_id}.")
        print(f"Error: {e}")

print(f"\n\n{'='*50}")
print("🎉 All models have been tested.")
print(f"{'='*50}")

In [2]:
minilm_embeddings = torch.load("/home/rjha5/603-nvme2/arena/index_wikipedia_sentence-transformers_all-MiniLM-L6-v2/embeddings.0.pt")

In [ ]:
minilm_embeddings.shape, minilm_embeddings.dtype, minilm_embeddings.device


(torch.Size([384, 3811232]), torch.bfloat16, device(type='cuda', index=0))

In [5]:
len(wikipedia)

2502070

In [6]:
import pickle

In [8]:
with open("/home/rjha5/603-nvme2/arena/index_wikipedia_sentence-transformers_all-MiniLM-L6-v2/passages.0.pt", "rb") as f:
    passages = pickle.load(f)
len(passages)



3811232

In [13]:
wikipedia = load_dataset("mteb/arena-wikipedia-7-15-24", split="train")

In [14]:
wikipedia

Dataset({
    features: ['title', 'id', 'text'],
    num_rows: 3811232
})

In [ ]:
from retrieval.common import CORPORA
formatted_passages = [CORPORA["wikipedia"]["format"].format(**p) for p in tqdm(wikipedia, total=len(wikipedia))]
formatted_passages[0]


In [20]:
model = SentenceTransformer("intfloat/multilingual-e5-small", trust_remote_code=True, device="cuda:0")

In [ ]:
x = model.encode(wikipedia["text"][:5_000], convert_to_tensor=True, show_progress_bar=True, batch_size=1024)

Batches:   0%|          | 0/98 [00:00<?, ?it/s]

KeyboardInterrupt: 